GSE173278: Glioblastoma 

source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE173278

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

In [1]:
!curl -sL -o GSE173278_filtered_meta.csv.gz "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE173nnn/GSE173278/suppl/GSE173278_scRNAseq_filtered_cells_metadata.csv.gz" && gunzip -k GSE173278_filtered_meta.csv.gz && curl -sL -o GSE173278_filtered_barcodes.tsv.gz "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE173nnn/GSE173278/suppl/GSE173278_scRNAseq_filtered_cells_barcodes.tsv.gz" && curl -sL -o GSE173278_filtered_genes.tsv.gz "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE173nnn/GSE173278/suppl/GSE173278_scRNAseq_filtered_cells_genes.tsv.gz" && echo "downl norm matrix" && curl -sL -o GSE173278_filtered_norm_counts.mtx.gz "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE173nnn/GSE173278/suppl/GSE173278_scRNAseq_filtered_cells_norm_counts_matrix.mtx.gz" && gunzip -k GSE173278_filtered_norm_counts.mtx.gz && echo "Done: $(wc -l < GSE173278_filtered_meta.csv) meta lines, $(du -h GSE173278_filtered_norm_counts.mtx | cut -f1) mtx"

downl norm matrix
Done: 75076 meta lines, 6.1G mtx


In [3]:
import os
from urllib.request import urlretrieve

BASE = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE173nnn/GSE173278/suppl/'
FILES = {
    'GSE173278_filtered_norm_counts.mtx.gz': 'GSE173278_scRNAseq_filtered_cells_norm_counts_matrix.mtx.gz',
    'GSE173278_filtered_barcodes.tsv.gz': 'GSE173278_scRNAseq_filtered_cells_barcodes.tsv.gz',
    'GSE173278_filtered_genes.tsv.gz': 'GSE173278_scRNAseq_filtered_cells_genes.tsv.gz',
    'GSE173278_filtered_meta.csv.gz': 'GSE173278_scRNAseq_filtered_cells_metadata.csv.gz',
}

for local, remote in FILES.items():
    if not os.path.exists(local):
        print(f'Downloading {local}...')
        urlretrieve(BASE + remote, local)
        print(f'  {os.path.getsize(local)/1e6:.1f} MB')

In [4]:
meta = pd.read_csv('GSE173278_filtered_meta.csv.gz', index_col=0, compression='gzip')
print(f'Metadata: {meta.shape[0]} cells')
print(meta['cell_type'].value_counts())

barcodes = pd.read_csv('GSE173278_filtered_barcodes.tsv.gz', header=None, compression='gzip')[0].values
genes = pd.read_csv('GSE173278_filtered_genes.tsv.gz', header=None, sep='\t', compression='gzip')[1].values
print(f'\nBarcodes: {len(barcodes)}, Genes: {len(genes)}')

print(f'Metadata cells match barcodes: {list(meta.index) == list(barcodes)}')

Metadata: 75075 cells
cell_type
malignant          67699
immune              4266
fibroblast          1296
endothelial          919
oligodendrocyte      803
neuron                35
Name: count, dtype: int64

Barcodes: 75075, Genes: 23076
Metadata cells match barcodes: True


In [5]:
import scipy.io
import gzip
import subprocess
mtx_file = 'GSE173278_filtered_norm_counts.mtx'
if not os.path.exists(mtx_file):
    subprocess.run(['gunzip', '-k', 'GSE173278_filtered_norm_counts.mtx.gz'], check=True)
    print(f'Decompressed: {os.path.getsize(mtx_file)/1e6:.0f} MB')

X = scipy.io.mmread(mtx_file).T.tocsr()  # transpose to cells × genes
print(f'Matrix: {X.shape[0]} cells × {X.shape[1]} genes')

adata = sc.AnnData(
    X=X.astype(np.float32),
    obs=meta.loc[barcodes],
    var=pd.DataFrame(index=genes)
)
adata.var_names_make_unique()
print(f'AnnData: {adata.shape}')

adata.obs['ground_truth_malignant'] = (adata.obs['cell_type'] == 'malignant').astype(int)
print(f'Malignant: {adata.obs["ground_truth_malignant"].sum()} / {len(adata)} ({adata.obs["ground_truth_malignant"].mean()*100:.1f}%)')

/tmp/ipykernel_633486/1787341100.py:12: DeprecationWarning: The default value for `spmatrix` is changing to `False` in v1.20.
             That means the default return type will be a sparse array.
             Unless you use * instead of @, ** for matrix power, or you depend
             on 2D shapes from e.g. `A.sum(axis=0)` it may not matter to you.
             See the spmatrix to sparray migration guide for details.
             https://docs.scipy.org/doc/scipy/reference/sparse.migration_to_sparray.html
             
  X = scipy.io.mmread(mtx_file).T.tocsr()  # transpose to cells × genes


Matrix: 75075 cells × 23076 genes
AnnData: (75075, 23076)
Malignant: 67699 / 75075 (90.2%)


/home1/prashantp/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [7]:
from aeacus import Profiler

aeacus_profiler = Profiler(
    test_input=adata.copy(),
    norm_type=False
)
aeacus_profiler.load()
aeacus_adata = aeacus_profiler.profile()

print('aeacus results:')
print(aeacus_adata.obs[['malignancy_call', 'malignancy_score']].describe())

malignant_mask = aeacus_adata.obs["malignancy_call"] == "Malignant"
n_malignant = malignant_mask.sum()
n_total = len(aeacus_adata)
percent = 100 * malignant_mask.mean()

print(f'\nCalled malignant: {n_malignant} / {n_total} ({percent:.1f}%)')

Model features: 3778
Missing features: 43 (1.14%)
aeacus results:
       malignancy_score
count      75075.000000
mean           0.890189
std            0.285482
min            0.002533
25%            0.981648
50%            0.990819
75%            0.994082
max            0.999381

Called malignant: 67655 / 75075 (90.1%)
